In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Cell 1 - Load all 4 CSVs from Lakehouse Files

df_properties = spark.read.option("header", True).option("inferSchema", True).csv("Files/properties.csv")
df_tenants = spark.read.option("header", True).option("inferSchema", True).csv("Files/tenants.csv")
df_payments = spark.read.option("header", True).option("inferSchema", True).csv("Files/rent_payments.csv")
df_maintenance = spark.read.option("header", True).option("inferSchema", True).csv("Files/maintenance_requests.csv")

print("Properties:", df_properties.count())
print("Tenants:", df_tenants.count())
print("Payments:", df_payments.count())
print("Maintenance:", df_maintenance.count())

StatementMeta(, 61a85c6d-54af-402a-998c-c55f85f54258, 4, Finished, Available, Finished, False)

Properties: 100
Tenants: 400
Payments: 4272
Maintenance: 600


In [3]:
# Cell 2 - Clean all 4 dataframes

from pyspark.sql.functions import col, trim, upper, to_date, when, regexp_replace

# --- Properties ---
df_properties_clean = df_properties \
    .dropDuplicates(["property_id"]) \
    .dropna(subset=["property_id", "city", "state"]) \
    .withColumn("state", upper(trim(col("state")))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("status", trim(col("status"))) \
    .withColumn("acquisition_date", to_date(col("acquisition_date"), "yyyy-MM-dd"))

# --- Tenants ---
df_tenants_clean = df_tenants \
    .dropDuplicates(["tenant_id"]) \
    .dropna(subset=["tenant_id", "property_id"]) \
    .withColumn("lease_start_date", to_date(col("lease_start_date"), "yyyy-MM-dd")) \
    .withColumn("lease_end_date", to_date(col("lease_end_date"), "yyyy-MM-dd")) \
    .withColumn("tenant_status", trim(col("tenant_status"))) \
    .withColumn("employment_status", trim(col("employment_status")))

# --- Payments ---
df_payments_clean = df_payments \
    .dropDuplicates(["payment_id"]) \
    .dropna(subset=["payment_id", "tenant_id"]) \
    .withColumn("due_date", to_date(col("due_date"), "yyyy-MM-dd")) \
    .withColumn("payment_date", to_date(col("payment_date"), "yyyy-MM-dd")) \
    .withColumn("payment_status", trim(col("payment_status"))) \
    .withColumn("days_late", when(col("days_late").isNull(), 0).otherwise(col("days_late")))

# --- Maintenance ---
df_maintenance_clean = df_maintenance \
    .dropDuplicates(["request_id"]) \
    .dropna(subset=["request_id", "property_id"]) \
    .withColumn("submitted_date", to_date(col("submitted_date"), "yyyy-MM-dd")) \
    .withColumn("resolved_date", to_date(col("resolved_date"), "yyyy-MM-dd")) \
    .withColumn("priority", trim(col("priority"))) \
    .withColumn("status", trim(col("status"))) \
    .withColumn("cost", when(col("cost").isNull(), 0).otherwise(col("cost")))

print("Properties clean:", df_properties_clean.count())
print("Tenants clean:", df_tenants_clean.count())
print("Payments clean:", df_payments_clean.count())
print("Maintenance clean:", df_maintenance_clean.count())

StatementMeta(, 61a85c6d-54af-402a-998c-c55f85f54258, 5, Finished, Available, Finished, False)

Properties clean: 100
Tenants clean: 400
Payments clean: 4272
Maintenance clean: 600


In [4]:
# Cell 3 - Save cleaned data as Delta tables

df_properties_clean.write.format("delta").mode("overwrite").saveAsTable("properties_clean")
df_tenants_clean.write.format("delta").mode("overwrite").saveAsTable("tenants_clean")
df_payments_clean.write.format("delta").mode("overwrite").saveAsTable("payments_clean")
df_maintenance_clean.write.format("delta").mode("overwrite").saveAsTable("maintenance_clean")

print("All 4 tables saved successfully!")

StatementMeta(, 61a85c6d-54af-402a-998c-c55f85f54258, 6, Finished, Available, Finished, False)

All 4 tables saved successfully!


In [7]:
# Fix month_year to proper date format

from pyspark.sql.functions import to_date, concat, lit

df_payments_clean = df_payments_clean.withColumn(
    "month_year", 
    to_date(concat(col("month_year"), lit("-01")), "yyyy-MM-dd")
)

df_payments_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("payments_clean")
print("Done!")


StatementMeta(, 61a85c6d-54af-402a-998c-c55f85f54258, 9, Finished, Available, Finished, False)

Py4JJavaError: An error occurred while calling o5354.saveAsTable.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 141.0 failed 4 times, most recent failure: Lost task 0.3 in stage 141.0 (TID 567) (vm-30592357 executor 2): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to abfss://9a61897c-0126-41e7-bbae-1770c4b78584@onelake.dfs.fabric.microsoft.com/0ebc8236-f409-4a6a-a543-4c9be5a91abf/Tables/dbo/payments_clean.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:777)
	at org.apache.spark.sql.delta.files.DeltaFileFormatWriter$.executeTask(DeltaFileFormatWriter.scala:621)
	at org.apache.spark.sql.delta.files.DeltaFileFormatWriter$.$anonfun$executeWrite$4(DeltaFileFormatWriter.scala:387)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:93)
	at org.apache.spark.TaskContext.runTaskWithListeners(TaskContext.scala:166)
	at org.apache.spark.sch

In [8]:
from pyspark.sql.functions import date_format, to_date, concat, lit, col

# Read the existing clean table
df_fix = spark.table("payments_clean")

# Add a readable month label like "Jan 23"
df_fix = df_fix.withColumn(
    "month_label",
    date_format(col("due_date"), "MMM yy")
)

df_fix.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("payments_clean")
print("Done!")

StatementMeta(, 61a85c6d-54af-402a-998c-c55f85f54258, 10, Finished, Available, Finished, False)

Done!
